# EdinburghAI: TinyML E-Nose

# Setup python environment

In [ ]:
# Setup environment
%pip install pandas numpy matplotlib ai-edge-litert tensorflow seaborn

# Constants

In [ ]:
FEATURE_COLS = ["temperature", "pressure", "humidity", "gas_resistance"]
FEATURE_COLS_EXTENDED = FEATURE_COLS + [f"{c}_diff" for c in FEATURE_COLS]
TARGET_COL = "label"

FILES = [
    "avocado_open_container.csv",
    # "eucalyptus_closed_container_15mins.csv",
    # "lavender_closed_container_15mins.csv",
    "open_room.csv",
    # "room_air_open_forced_15mins.csv",
    "rose_open_container.csv",
    "rose_closed_container.csv",
]

# Load Data

In [ ]:
import pandas as pd
import os

dfs = []
for file in os.listdir("../data/custom"):
    if file not in FILES: continue

    df = pd.read_csv(f"../data/custom/{file}")
    dfs.append(df)

NUM_CLASSES = len(dfs)

data = pd.concat(dfs)
print(len(data), "entries")
data.head()

# Graph Data

In [ ]:
# import matplotlib.pyplot as plt
# import seaborn as sns

# plt.figure(figsize=(8,6))

# datas = data.sample(n=1000)  # sample so it doesn't take 10 years
# sns.pairplot(datas[FEATURE_COLS+[TARGET_COL]], hue=TARGET_COL, palette='Set1', diag_kind='hist')
# plt.show()


# Train Neural Network

## Split data into training and testing

In [ ]:
TESTING_RATIO = 0.9

training_data = data.sample(frac=(1-TESTING_RATIO), random_state=67)
testing_data = data.drop(training_data.index)

print(f"{len(training_data)} training : {len(testing_data)} testing samples")

## Data Preprocessing

In [ ]:
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler

x_train = training_data[FEATURE_COLS]
y_train = training_data[TARGET_COL]

x_test = testing_data[FEATURE_COLS]
y_test = testing_data[TARGET_COL]


label_encoder = LabelEncoder()

y_train_enc = label_encoder.fit_transform(y_train)
y_test_enc = label_encoder.transform(y_test)

scaler = StandardScaler()

x_train = scaler.fit_transform(x_train)
x_test = scaler.transform(x_test)

x_train[:, 3] = np.log1p(x_train[:, 3])
x_test[:, 3] = np.log1p(x_test[:, 3])

## Build model

In [ ]:
from keras.models import Sequential
from keras.layers import Dense, Activation, Dropout, Input

model = Sequential()

model.add(Input(shape=(4,)))

model.add(Dense(32, activation='relu'))
model.add(Dropout(0.2))

model.add(Dense(32, activation='relu'))
model.add(Dropout(0.2))

model.add(Dense(NUM_CLASSES, activation='softmax'))

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

## Train Model

In [ ]:
from keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

history = model.fit(
    x_train,
    y_train_enc,
    validation_split=0.2,
    epochs=200,
    batch_size=64,
    callbacks=[early_stop]
)

## Graph the loss

In [ ]:
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (10,5)

loss = history.history['loss']
val_loss = history.history['val_loss']
epochs = range(1, len(loss) + 1)
plt.plot(epochs, loss, 'g.', label='Training loss')
plt.plot(epochs, val_loss, 'b', label='Validation loss')
plt.title('Training and validation loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.show()

print(plt.rcParams["figure.figsize"])

# Run with test data

In [ ]:
test_loss, test_acc = model.evaluate(x_test, y_test_enc)
print("Test accuracy:", test_acc)

In [ ]:
from collections import defaultdict

# Track correct + total per class
class_correct = defaultdict(int)
class_total = defaultdict(int)

# Use the Keras model (faster than LiteRT for evaluation)
y_pred_probs = model.predict(x_test)
y_pred = np.argmax(y_pred_probs, axis=1)

for true, pred in zip(y_test_enc, y_pred):
    class_total[true] += 1
    if true == pred:
        class_correct[true] += 1

# Print per-class accuracy
print("\nPer-class accuracy:")
for class_idx in sorted(class_total.keys()):
    class_name = label_encoder.inverse_transform([class_idx])[0]
    acc = class_correct[class_idx] / class_total[class_idx]
    print(f"{class_name:20s} | {acc:.4f} ({class_correct[class_idx]}/{class_total[class_idx]})")

# Confusion Matrix

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

# Predictions
y_pred_probs = model.predict(x_test)
y_pred = np.argmax(y_pred_probs, axis=1)

# Confusion matrix (counts)
cm = confusion_matrix(y_test_enc, y_pred)

# Convert to percentages (row-wise)
cm_percent = cm.astype('float') / cm.sum(axis=1, keepdims=True) * 100

# Handle any divide-by-zero (just in case)
cm_percent = np.nan_to_num(cm_percent)

# Class labels
class_names = label_encoder.classes_

# Plot
plt.figure(figsize=(8,6))
sns.heatmap(
    cm_percent,
    annot=True,
    fmt=".1f",
    xticklabels=class_names,
    yticklabels=class_names
)

plt.xlabel("Predicted label")
plt.ylabel("True label")
plt.title("Confusion Matrix (%)")
plt.show()

# Convert to litert model

In [ ]:
import tensorflow as tf
import os

converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]  # quantization
tflite_model = converter.convert()

with open('model.tflite', 'wb') as f:
  f.write(tflite_model)

basic_model_size = os.path.getsize("model.tflite")
print("Model is %d bytes" % basic_model_size)

## Test litert model

In [ ]:
from ai_edge_litert.compiled_model import CompiledModel

model_lite = CompiledModel.from_file("model.tflite")

signature_index = 0

input_buffers = model_lite.create_input_buffers(signature_index)
output_buffers = model_lite.create_output_buffers(signature_index)


n_correct = 0

for i in range(len(x_test)):
    input_data = np.float32(x_test[i])
    input_buffers[0].write(input_data)

    model_lite.run_by_index(signature_index, input_buffers, output_buffers)
    output_array = output_buffers[0].read(NUM_CLASSES, np.float32)

    pred_class = int(np.argmax(output_array))
    true_class = int(y_test_enc[i])

    if pred_class == true_class:
        n_correct += 1

print("LiteRT accuracy:", n_correct / len(x_test))

## Encode the model in an arduino header file

In [ ]:
with open("model.tflite", "rb") as f:
    data = f.read()

with open("model.h", "w") as f:
    f.write("const unsigned char model[] = {\n")
    
    for i, byte in enumerate(data):
        f.write(f"0x{byte:02x},")
        if (i + 1) % 12 == 0:
            f.write("\n")
    
    f.write("\n};\n")
    f.write(f"const unsigned int model_len = {len(data)};\n")

import os
print(f"Header file, model.h, is {os.path.getsize('model.h'):,} bytes.")